This notebook aims to label cell types using a semi-supervised classification algorithm PhenoGraph. 

By: Britney Tieu Forsyth

In [55]:
# Import packages
from typing import Optional
import phenograph
import scanpy as sc
from anndata import AnnData
from anndata import concat
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib
import os
import pandas as pd
import numpy as np
from scipy.stats import entropy
import subprocess
import palantir
import sys
import random
from collections import OrderedDict
import re
from itertools import chain
import warnings
sys.path.append('/lila/home/forsythb/anaconda3/envs/scrna/lib/python3.8/site-packages')
import harmony
from scipy.sparse import coo_matrix
from sklearn.neighbors import NearestNeighbors
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
%matplotlib inline

# Dataset Preparation

In [56]:
adata_base = sc.read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/adatas/Tumor_Epithelial_KG146_OKG146Li_Base.h5ad")

In [57]:
# In adata for base media, extract the rows that are not labeled 'Organoid' in the ['Cell State'] column
adata_patient = adata_base[adata_base.obs['Cell State'] != 'Organoid', ]

In [68]:
# Extract row names from the AnnData object
row_names = adata_organoid.obs_names

# Add 'Tumor Site' column based on conditions
adata_organoid.obs['Tumor_Site'] = ['Primary' if '146P' in name else 'Metastatic' if '146Li' in name else 'Unknown' for name in row_names]
adata_organoid.obs['Culture_Media'] = ['BASE' if '_BASE_' in name else 'HISC' if '_HISC_' in name else 'Dedifferentiated' if '_dedifferentiation_' in name else 'Unknown' for name in row_names]
adata_organoid.obs['ZFP_Expression'] = ['CTRL' if 'shC' in name else 'ZFPKD' if 'shZ' in name else 'Unknown' for name in row_names]


In [69]:
adata_organoid.obs

,background_fraction,cell_probability,cell_size,droplet_efficiency,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,...,pct_counts_in_top_500_genes,log10GenesPerUMI,Sample,original_total_counts,log10_original_total_counts,mito_frac,Tumor Site,Tumor_Site,Culture_Media,ZFP_Expression
146P_HISC_shCTRL_CCGGACACACCACATA-1,0.022291,0.999955,14050.672852,1.324225,4761,8.468423,16009,9.680969,40.983197,44.368793,...,58.267225,0.874733,146P_HISC_shCTRL,16009,4.204364,0.150103,Primary,Primary,HISC,CTRL
146P_dedifferentiation_shZFP36L2_3_ACTTTGTTCGCTGTTC-1,0.051227,0.999950,14127.892578,1.150659,3471,8.152486,12909,9.465757,42.389031,48.989077,...,67.084979,0.861237,146P_dedifferentiation_shZFP36L2_3,12909,4.110893,0.040127,Primary,Primary,Dedifferentiated,ZFPKD
KG146Li_BASE_shZFP36L2_3_GTGGTTAGTTGGGAAC-1,0.007611,0.999955,14619.753906,2.005845,5775,8.661467,26207,10.173820,36.402488,44.014958,...,60.777655,0.851335,KG146Li_BASE_shZFP36L2_3,26207,4.418417,0.059221,Metastatic,Metastatic,BASE,ZFPKD
146P_dedifferentiation_shCtrl_GGGTCTGTCGATGCTA-1,0.032237,0.999955,15622.842773,1.357936,4567,8.426831,17502,9.770128,39.389784,45.829048,...,62.987087,0.862492,146P_dedifferentiation_shCtrl,17502,4.243088,0.084676,Primary,Primary,Dedifferentiated,CTRL
146P_HISC_shZFP36L2_3_TGGAACTCAACCTAAC-1,0.013318,0.999955,15020.916016,1.453675,5019,8.521185,19114,9.858229,30.375641,39.546929,...,58.250497,0.864357,146P_HISC_shZFP36L2_3,19114,4.281352,0.065816,Primary,Primary,HISC,ZFPKD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146Li_dedifferentiation_shZFP36L2_4_AAATGGAAGCGGTAAC-1,0.018784,0.999946,15664.033203,1.131694,3917,8.273337,15358,9.639457,46.627165,53.223076,...,68.316187,0.858258,146Li_dedifferentiation_shZFP36L2_4,15358,4.186335,0.081847,Metastatic,Metastatic,Dedifferentiated,ZFPKD
146P_BASE_shZFP36L2_3_TCGTGGGAGCTCTTCC-1,0.056776,1.000000,8108.023438,0.866909,2184,7.689371,5582,8.627482,37.782157,48.262272,...,67.860982,0.891230,146P_BASE_shZFP36L2_3,5582,3.746790,0.059835,Primary,Primary,BASE,ZFPKD
146Li_dedifferentiation_shCtrl_TAGACCAAGTAGACCG-1,0.022374,0.999881,14368.451172,1.182307,3695,8.215006,14463,9.579418,47.735601,55.458757,...,69.591371,0.857546,146Li_dedifferentiation_shCtrl,14463,4.160258,0.072461,Metastatic,Metastatic,Dedifferentiated,CTRL
146P_dedifferentiation_shCtrl_GGGCGTTAGAAGGCTC-1,0.092006,0.999999,11828.569336,0.860723,2583,7.857094,7451,8.916238,44.584620,50.328815,...,67.628506,0.881182,146P_dedifferentiation_shCtrl,7451,3.872215,0.103342,Primary,Primary,Dedifferentiated,CTRL


In [70]:
# Extract the rows in the metacell adata where Culture_Media == BASE
adata_organoid = adata_metacells[adata_metacells.obs['Culture_Media'] == "BASE", ]

adata_organoid.obs['Cell State'] = pd.NA
# Create another column called ['Cell State']
# Fill in thhose values as NA

/scratch/lsftmp/3256154.tmpdir/ipykernel_129941/308119695.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_organoid.obs['Cell State'] = pd.NA


In [2]:
# This is the adata that contains samples from patient and patient organoids 
adata_patient_organoid = sc.read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/adatas/KG146_Patient_Organoid.h5ad")

In [18]:
# We want to use the labeled, patient data to generate labels for the organoid data
# These are samples with '_KG146M' in the names
adata_patient = adata_patient_organoid[adata_patient_organoid.obs_names.str.contains('_KG146M')].copy()

# We don't want to label the cells as TA-like, so let's remove rows with those labels
adata_patient = adata_patient[adata_patient.obs['Cell State'] != 'TA-like'].copy()

In [19]:
# Ok. We removed the TA-like cell labels. 
adata_patient.obs['Cell State']

120703424285939_KG146M            ISC-like
120703436155741_KG146M                 SCC
120703455025013_KG146M            ISC-like
120718456679846_KG146M    Fetal Progenitor
120718468987109_KG146M     Enterocyte-like
                                ...       
241109193414900_KG146M       Injury Repair
241109220584862_KG146M                 SCC
241176048225691_KG146M            ISC-like
241176061270774_KG146M         Goblet-like
241184516782309_KG146M                 SCC
Name: Cell State, Length: 1224, dtype: category
Categories (7, object): ['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', 'ISC-like', 'Injury Repair', 'SCC']

In [4]:
# This is the organoid adata that I've been using for the entire analysis pipeline.
adata_organoid = sc.read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/combined/out_post_hvg/adata.filteredmultiplex.combined.hvg_5000.h5ad")

In [8]:
# We want to label the patient 146 organoid data
adata_organoid = adata_organoid[adata_organoid.obs_names.str.contains('146')]

In [20]:
# Make a new .obs column in adata_organoid called ['Cell State']
# This should be filled in with NA values. 
adata_organoid.obs['Cell State'] = pd.NA

/scratch/lsftmp/3045571.tmpdir/ipykernel_96098/2277901049.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_organoid.obs['Cell State'] = pd.NA


In [71]:
'''
1. Let's concatenate the patient adata and organoid adata based on similar gene names. 
2. Then, create a dataframe to organize the count matrix, cell names, and gene names. 
'''
# Ensure both objects have the same set of genes
common_genes = adata_patient.var_names.intersection(adata_organoid.var_names)

# Subset both objects to include only common genes
adata_patient_common = adata_patient[:, common_genes].copy()
adata_organoid_common = adata_organoid[:, common_genes].copy()

# Concatenate the data using anndata.concat
concatenated_data = concat([adata_patient_common, adata_organoid_common])

# Extract count matrix
counts_patient_organoid = pd.DataFrame(concatenated_data.X.toarray(),
                                       index=concatenated_data.obs_names,
                                       columns=concatenated_data.var_names)

/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/merge.py:1263: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/merge.py:1348: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


In [72]:
# Reprocess the data
# 1. Normalization
norm_df = harmony.utils.normalize_counts(counts_patient_organoid)

# Gene selection
hvg_genes = harmony.utils.hvg_genes(norm_df)

# Log transform
data_df = harmony.utils.log_transform(norm_df.loc[:,hvg_genes])

/lila/home/forsythb/anaconda3/envs/scrna/lib/python3.8/site-packages/harmony/utils.py:81: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby('mean_bin')['dispersion']


In [73]:
data_df

,MT-CO2,MT-CO3,MT-ATP6,DEFA6,MT-CO1,ROBO2,MT-ND3,MT-CYB,MALAT1,DEFA5,...,NECTIN4,TUBB,BMX,AKT3,APOBEC3C,SPINK13,SULT1C4,PRSS2,EEF1A2,CARMN
120703424285939_KG146M,3.603694,3.563984,3.336051,-3.321928,3.489318,-3.321928,3.256771,3.273608,3.390256,-3.321928,...,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
120703436155741_KG146M,3.362617,3.317239,3.201443,-3.321928,3.284338,0.678852,3.076038,3.031388,3.754169,2.217832,...,1.754399,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,0.678852,-3.321928,1.999391,-3.321928
120703455025013_KG146M,3.290662,3.183425,3.069695,-3.321928,3.210814,-3.321928,2.843803,3.311974,3.379315,0.449374,...,-3.321928,0.449374,-3.321928,-3.321928,-0.203058,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
120718456679846_KG146M,3.836418,3.842934,3.779542,2.617080,3.779542,-3.321928,3.713533,3.826235,3.987879,3.371298,...,-3.321928,2.228768,-3.321928,-3.321928,-3.321928,-3.321928,2.228768,-3.321928,-3.321928,-3.321928
120718468987109_KG146M,3.772816,3.749225,3.551318,-3.321928,3.851496,-3.321928,3.493898,3.643136,3.726367,-3.321928,...,-3.321928,-3.321928,-3.321928,-3.321928,1.999232,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146_P_BASE_ZFPKD_2_SEACell-102,7.019153,6.790813,6.136559,-3.321928,5.757308,-3.321928,6.358901,5.318002,5.719382,-3.321928,...,-3.321928,3.415081,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
146_M_BASE_ZFPKD_1_SEACell-2,7.613805,7.502404,7.243769,5.211630,6.958743,-2.467005,5.449129,6.380441,6.320394,-0.987557,...,-3.321928,0.910009,-3.321928,-2.467005,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
146_M_BASE_ZFPKD_1_SEACell-46,7.428383,7.096501,6.648559,-0.060465,6.779000,-2.783806,5.207294,6.011711,6.688312,-3.321928,...,-2.783806,2.632990,-3.321928,-2.783806,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
146_M_BASE_ZFPKD_2_SEACell-76,9.688032,9.424044,9.033275,-3.321928,8.769258,-3.321928,7.229717,8.217411,7.274301,-3.321928,...,-3.321928,-1.068339,-3.321928,-1.793687,-3.321928,-3.321928,-1.793687,-3.321928,-3.321928,-3.321928


## Get Affinities

In [74]:
# The patient data is defined as those cells with 'KG146M' in the row name
# Define the time point connections
tp = pd.Series(index=data_df.index)

# Define pattern for Patient cells
patient_pattern = 'KG146M'

# Use str.contains to check if the patient pattern is present in the cell identifier
patient_cells = data_df.index[data_df.index.str.contains(patient_pattern)]

# Assign 'Patient' to the corresponding cells in the series
tp[patient_cells] = 'Patient'

# The remaining cells (not classified as Patient) are Organoid cells
tp.fillna('Organoid', inplace=True)

organoid_count = (tp == 'Organoid').sum()
patient_count = (tp == 'Patient').sum()

print(f"Number of Organoid cells: {organoid_count}")
print(f"Number of Patient cells: {patient_count}")

Number of Organoid cells: 2178
Number of Patient cells: 1304


/scratch/lsftmp/3256154.tmpdir/ipykernel_129941/3338450720.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Patient' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tp[patient_cells] = 'Patient'


In [75]:
# Define timepoint connections
timepoint_connections = pd.DataFrame(columns=[0, 1])
index = 0
timepoint_connections.loc[index, :] = ['Organoid', 'Patient']; index += 1
timepoint_connections

,0,1
0,Organoid,Patient


In [76]:
'''
Harmony and Palantir
'''
def _mnn_ka_distances(mnn, n_neighbors):
    # Function to find distance ka^th neighbor in the mutual nearest neighbor matrix
    ka = int(n_neighbors / 3)
    ka_dists = np.repeat(None, mnn.shape[0])
    x, y, z = find(mnn)
    rows=pd.Series(x).value_counts()
    for r in rows.index[rows >= ka]:
        ka_dists[r] = np.sort(z[x==r])[ka - 1]
    return ka_dists

#######################################################################################
from harmony import core
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import find, csr_matrix

def harmony_aug_mat_with_pca(projections, timepoints, timepoint_connections):
    # Time point cells and indices
    tp_cells = pd.Series()
    tp_offset = pd.Series()
    offset = 0
    for i in timepoints.unique():
        tp_offset[i] = offset
        tp_cells[i] = list(timepoints.index[timepoints == i])
        offset += len(tp_cells[i])
    n_neighbors = 30
    
    # Nearest neighbor graph construction and affinity matrix
    print('Nearest neighbor computation...')
    nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                            metric='cosine', n_jobs=-2).fit(projections.values)

    adj = nbrs.kneighbors_graph(projections.values, mode='distance')
    dists, _ = nbrs.kneighbors(projections.values)
    
    # Scaling factors for affinity matrix construction
    ka = int(n_neighbors / 3)
    scaling_factors = pd.Series(dists[:, ka], index=projections.index)
    
    # Affinity matrix
    nn_aff = core._convert_to_affinity(adj, scaling_factors, True)
    n_jobs = -2
    
    # Mututally nearest neighbor affinity matrix
    # Initilze mnn affinity matrix
    N = projections.shape[0]
    full_mnn_aff = csr_matrix(([0], ([0], [0])), [N, N])
    for i in timepoint_connections.index:
        t1, t2 = timepoint_connections.loc[i, :].values
        print(f'Constucting affinities between {t1} and {t2}...')
        # MNN matrix  and distance to ka the distance
        t1_cells = tp_cells[t1]
        t2_cells = tp_cells[t2]
        mnn = core._construct_mnn(t1_cells, t2_cells, projections,
                             n_neighbors, n_jobs)
        
        # MNN Scaling factors
        # Distance to the adaptive neighbor
        ka_dists = pd.Series(0.0, index=t1_cells + t2_cells)
        ka_dists = ka_dists.astype(float)
        # T1 scaling factors
        ka_dists[t1_cells] = _mnn_ka_distances(mnn, n_neighbors)
        # T2 scaling factors
        ka_dists[t2_cells] = _mnn_ka_distances(mnn.T, n_neighbors)
        # Scaling factors
        mnn_scaling_factors = pd.Series(0.0, index=projections.index)
        mnn_scaling_factors[t1_cells] = core._mnn_scaling_factors(
            ka_dists[t1_cells], scaling_factors)
        mnn_scaling_factors[t2_cells] = core._mnn_scaling_factors(
            ka_dists[t2_cells], scaling_factors)
        # MNN affinity matrix
        full_mnn_aff = full_mnn_aff + \
            core._mnn_affinity(mnn, mnn_scaling_factors,
                          tp_offset[t1], tp_offset[t2])
    # Symmetrize the affinity matrix and return
    aug_aff2 = nn_aff + nn_aff.T + full_mnn_aff + full_mnn_aff.T
    aff2 = nn_aff + nn_aff.T
    return aug_aff2, aff2

In [77]:
pca_merge = pd.DataFrame(concatenated_data.obsm['X_pca'], index=concatenated_data.obs_names)
aug_mat, aff_mat = harmony_aug_mat_with_pca(pca_merge, tp, timepoint_connections)

KeyError: 'X_pca'

In [34]:
# Convert arrays to string format
# Convert non-string columns in the "obs" DataFrame to strings
concatenated_data.obs = concatenated_data.obs.applymap(str)

# Add aug_mat to adata_patient_organoid
concatenated_data.obsm['aug_mat'] = aug_mat.toarray()

# Add aff_mat to adata_patient_organoid
concatenated_data.obsm['aff_mat'] = aff_mat.toarray()

# Save the modified patient 146 organoid adata 
concatenated_data.write('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/phenograph/adata_146_organoid_harmony.h5ad')

/scratch/lsftmp/3256154.tmpdir/ipykernel_129941/1408172278.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  concatenated_data.obs = concatenated_data.obs.applymap(str)


## PhenoGraph Classification

In [35]:
adata_patient_organoid = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/phenograph/adata_146_organoid_harmony.h5ad')

In [36]:
aug_mat = pd.DataFrame(adata_patient_organoid.obsm['aug_mat'], index=adata_patient_organoid.obs.index, columns = adata_patient_organoid.obs.index)

In [37]:
adata_patient_organoid.obs['Cell State'] = adata_patient_organoid.obs['Cell State'].astype(str).fillna('NaN')

In [38]:
ind = adata_patient_organoid.obs['Cell State'] != 'NaN'
adata_patient_organoid = adata_patient_organoid[ind, :]
aug_mat = aug_mat.loc[ind, ind]

In [39]:
# Select cells with labeled cell state (excluding 'NA')
ind = adata_patient_organoid.obs['Cell State'] != 'nan'
labeled_cells = adata_patient_organoid[ind,:]

# Get unique cell types/categories
cell_types = labeled_cells.obs['Cell State'].unique()

# Initialize an empty list to store training data for each cell type
train_data = []
train_labels = []
ct_codes = []

# Iterate over cell types and extract 'aug_mat' for each
for i,cell_type in enumerate(cell_types):
    # Filter cells of the current cell type
    cells_of_type = labeled_cells[labeled_cells.obs['Cell State'] == cell_type]
    
    # Extract 'aug_mat' for these cells
    pc_data = cells_of_type.obsm['X_pca']  # Assuming sparse matrix to array
    
    # Append 'aug_mat' data to the list
    train_data.append(pc_data)
    train_labels.append(cells_of_type.obs.index.to_series())   
    ct_codes += [i+1]*pc_data.shape[0]

In [40]:
train_labels = pd.concat(train_labels).index

In [41]:
# # Select cells with labeled cell state (excluding 'nan') for test data
test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] == 'nan']
#adata_patient_organoid.obs['Cell State'] = adata_patient_organoid.obs['Cell State'].replace('NaN', np.nan)
#test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'].isna()]

# # Extract 'aug_mat' for test cells
test_data = test_cells.obsm['X_pca']
test_labels = test_cells.obs.index

In [42]:
labels = pd.Index(test_labels.tolist() +  train_labels.tolist())

In [43]:
aug_all = coo_matrix(aug_mat.loc[labels,labels].values )

In [44]:
ct_codes = np.array([0]*len(test_labels) + ct_codes)

In [45]:
def preprocess(train, test):
    labels = np.zeros((test.shape[0], ), dtype=int)
    data = test
    for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data = np.append(data, examples, axis=0)
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    return data, labels

In [46]:
data, labels = preprocess(train_data, test_data)

In [47]:
def create_graph(data, k=30, metric='euclidean', n_jobs=-1):
    # def _kernel(dxy, sigma=1):
    #     return np.exp(-dxy ** 2 / sigma)

    _, idx = find_neighbors(data, k=k, metric=metric, n_jobs=n_jobs)
    # affinities = np.apply_along_axis(lambda x: _kernel(x, x.std()), axis=1, arr=d)
    # n, k = idx.shape
    # i = [np.tile(x, (k, )) for x in range(n)]
    # i = np.concatenate(np.array(i))
    # j = np.concatenate(idx)
    # s = np.concatenate(affinities)
    # graph = sp.coo_matrix((s, (i, j)), shape=(n, n)).tocsr()
    # graph = (graph + graph.transpose()).multiply(.5)
    graph = neighbor_graph(jaccard_kernel, {'idx': idx})
    # make symmetric
    # graph = (graph + graph.transpose()).multiply(.5)
    return graph

In [48]:
# Assuming aug_all is your coo_matrix
subset_rows = slice(0, 50)
subset_cols = slice(0, 10)
subset_aug_all = aug_all.tocsr()[subset_rows, subset_cols]

# Now pass the subset to the create_graph function
# create_graph(subset_aug_all)

# dense_subset_aug_all = subset_aug_all.toarray()
# create_graph(dense_subset_aug_all)

In [49]:
dense_subset_aug_all = subset_aug_all.toarray()
create_graph(dense_subset_aug_all)

Finding 30 nearest neighbors using minkowski metric and 'auto' algorithm


<50x50 sparse matrix of type '<class 'numpy.float64'>'
	with 1500 stored elements in COOrdinate format>

In [50]:
import numpy as np
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
import scipy.sparse as sp
from sklearn.preprocessing import normalize


def random_walk_probabilities(A, labels):

    if labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()
        D = sp.diags(A.sum(axis=1).flatten(), [0], shape=A.shape, format='csr')
        L = D - A
        seeds = labels != 0
        Lu = L[np.invert(seeds), :]  # unlabeled rows
        Lu = Lu.tocsc()[:, np.invert(seeds)]  # unlabeled columns
        # Check that Lu has the right size
        if not all([t == sum(np.invert(seeds)) for t in Lu.shape]):
            raise IndexError("Lu should be square and match size of test data")
        BT = L[np.invert(seeds), :]  # unlabeled rows
        BT = BT.tocsc()[:, seeds]  # labeled columns
        if not (sum(np.invert(seeds)), sum(seeds)) == BT.shape:
            raise IndexError("BT size is incorrect")
        # Matrix representation of labels
        i, j, s = [], [], []
        classes = np.unique(labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(labels[seeds] == k)[0])
            j.extend(np.tile(k, sum(seeds == k)))
            s.extend(np.tile(1, sum(seeds == k)))
        i = np.arange(seeds.sum())
        j = labels[seeds] - 1
        s = np.ones((seeds.sum(), ))
        M = sp.coo_matrix((s, (i, j)), shape=(seeds.sum(), len(classes))).tocsc()
        # P = sp.linalg.spsolve(Lu.tocsc(), -BT.dot(M))
        # Use iterative solver
        B = -BT.dot(M)
        vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]
        warnings = [x[1] for x in vals]
        if any(warnings):
            print("Warning: iterative solver failed to converge in at least one case", flush=True)
        P = normalize(np.vstack(tuple((x[0] for x in vals))).T, norm='l1')

    else:
        D = np.diag(np.sum(A, axis=1))
        L = D - A  # graph laplacian
        seeds = np.array([e != 0 for e in labels], dtype=bool)
        Lu = L[seeds, :][:, seeds]  # labeled rows, labeled cols
        BT = L[~seeds, :][:, seeds]  # unlabeled rows, labeled cols
        classes = np.unique(labels[labels > 0])
        M = np.zeros((seeds.sum(), len(classes)))
        for k in classes:
            M[labels[seeds] == k, k] = 1
        P = np.linalg.lstsq(Lu, np.dot(-BT, M))[0]

    return P

In [51]:
P = random_walk_probabilities(aug_all, ct_codes)

/scratch/lsftmp/3256154.tmpdir/ipykernel_129941/3655095487.py:42: DeprecationWarning: Please use `bicgstab` from the `scipy.sparse.linalg` namespace, the `scipy.sparse.linalg.isolve` namespace is deprecated.
  vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]


In [52]:
pval_df = pd.DataFrame(P, index = test_cells.obs_names, columns = cell_types)

In [53]:
pval_df.to_csv('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/phenograph/classification_organoid_base.csv', index=True)

In [54]:
pval_df

,ISC,Squamous,Fetal,Absorptive,Proliferative,Injury Repair,Neuroendocrine,Secretory
146_P_BASE_ZFPKD_2_SEACell-139,0.051089,0.133322,0.005403,0.276981,0.400681,0.025030,0.0,0.107493
146_P_BASE_ZFPKD_1_SEACell-90,0.048298,0.174516,0.007019,0.262848,0.371723,0.032227,0.0,0.103370
125_P_BASE_ZFPKD_2_SEACell-29,0.014176,0.539000,0.000810,0.192605,0.031403,0.213333,0.0,0.008673
146_P_BASE_CTRL_1_SEACell-121,0.050178,0.145999,0.005874,0.275088,0.387825,0.027508,0.0,0.107529
125_P_BASE_ZFPKD_1_SEACell-120,0.013545,0.442844,0.000728,0.304910,0.029269,0.200603,0.0,0.008102
...,...,...,...,...,...,...,...,...
146_P_BASE_ZFPKD_2_SEACell-63,0.048523,0.170552,0.006789,0.265252,0.372956,0.031998,0.0,0.103930
146_P_BASE_CTRL_1_SEACell-204,0.040315,0.312508,0.014420,0.203471,0.307558,0.041239,0.0,0.080489
146_P_BASE_ZFPKD_2_SEACell-56,0.046986,0.198810,0.008463,0.250007,0.363293,0.033151,0.0,0.099289
125_P_BASE_CTRL_1_SEACell-31,0.017747,0.468180,0.001084,0.250926,0.045128,0.204416,0.0,0.012518


In [6]:
adata_patient_organoid.obs

,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,original_total_counts,log10_original_total_counts,Cell State,Patient
120703424285939_KG146M,4122,8.324336,12791.0,9.456575,35.188805,40.575405,47.416152,58.603706,12791.0,4.106904,ISC-like,NaN
120703436155741_KG146M,4282,8.362409,20657.0,9.935858,33.862613,42.358523,51.299802,64.660890,20657.0,4.315067,SCC,NaN
120703455025013_KG146M,7117,8.870382,38341.0,10.554301,36.214496,42.794919,49.894369,59.197726,38341.0,4.583663,ISC-like,NaN
120718456679846_KG146M,1939,7.570443,4448.0,8.400435,27.585432,37.230216,47.819245,64.568345,4448.0,3.648165,Fetal Progenitor,NaN
120718468987109_KG146M,2948,7.989221,9450.0,9.153876,37.947090,46.814815,55.544974,67.280423,9450.0,3.975432,Enterocyte-like,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
125P_HISC_shZFP36L2_4_TTCGGTCTCCGTTGGG-1,3433,8.141481,10813.0,9.288597,33.764913,44.696199,53.333950,64.311477,10813.0,4.033946,nan,125.0
146P_HISC_shZFP36L2_4_ACAAAGATCATCACTT-1,4342,8.376321,11932.0,9.387063,30.263158,36.188401,42.675159,53.863560,11932.0,4.076713,nan,146.0
146P_BASE_shZFP36L2_3_TCAGGTAAGTTTAGGA-1,2957,7.992269,8241.0,9.016998,31.707317,41.730373,50.661328,63.074870,8241.0,3.915980,nan,146.0
146P_HISC_shZFP36L2_3_TCATCATAGATCGGTG-1,4366,8.381832,14735.0,9.598049,30.580251,40.190024,48.408551,59.748897,14735.0,4.168350,nan,146.0


In [7]:
# Select cells with labeled cell state (excluding 'NA')
ind = adata_patient_organoid.obs['Cell State'] != 'nan'
labeled_cells = adata_patient_organoid[ind,:]

# Get unique cell types/categories
cell_types = labeled_cells.obs['Cell State'].unique()

# Initialize an empty list to store training data for each cell type
train_data = []
train_labels = []
ct_codes = []

# Iterate over cell types and extract 'aug_mat' for each
for i,cell_type in enumerate(cell_types):
    # Filter cells of the current cell type
    cells_of_type = labeled_cells[labeled_cells.obs['Cell State'] == cell_type]
    
    # Extract 'aug_mat' for these cells
    pc_data = cells_of_type.obsm['X_pca']  # Assuming sparse matrix to array
    
    # Append 'aug_mat' data to the list
    train_data.append(pc_data)
    train_labels.append(cells_of_type.obs.index.to_series())   
    ct_codes += [i+1]*pc_data.shape[0]

In [8]:
labeled_cells

View of AnnData object with n_obs × n_vars = 1304 × 16622
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'original_total_counts', 'log10_original_total_counts', 'Cell State', 'Patient'
    obsm: 'X_pca', 'X_umap', 'aff_mat', 'aug_mat'

In [9]:
train_labels = pd.concat(train_labels).index

In [10]:
train_labels

Index(['120703424285939_KG146M', '120703455025013_KG146M',
       '120786758323435_KG146M', '120849035352805_KG146M',
       '121254239168811_KG146M', '121405261146526_KG146M',
       '121735021448429_KG146M', '121877975481140_KG146M',
       '121955392276916_KG146M', '121955407223731_KG146M',
       ...
       '231365435575662_KG146M', '232250957257508_KG146M',
       '234992267057070_KG146M', '235060820065076_KG146M',
       '235060851600300_KG146M', '235127558224605_KG146M',
       '239476046425454_KG146M', '240559720717021_KG146M',
       '241097677924214_KG146M', '241176061270774_KG146M'],
      dtype='object', length=1304)

In [11]:
# # Select cells with labeled cell state (excluding 'nan') for test data
test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] == 'nan']
#adata_patient_organoid.obs['Cell State'] = adata_patient_organoid.obs['Cell State'].replace('NaN', np.nan)
#test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'].isna()]

# # Extract 'aug_mat' for test cells
test_data = test_cells.obsm['X_pca']
test_labels = test_cells.obs.index

In [12]:
adata_patient_organoid

View of AnnData object with n_obs × n_vars = 53963 × 16622
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'original_total_counts', 'log10_original_total_counts', 'Cell State', 'Patient'
    obsm: 'X_pca', 'X_umap', 'aff_mat', 'aug_mat'

In [13]:
test_cells

View of AnnData object with n_obs × n_vars = 52659 × 16622
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'original_total_counts', 'log10_original_total_counts', 'Cell State', 'Patient'
    obsm: 'X_pca', 'X_umap', 'aff_mat', 'aug_mat'

In [14]:
labels = pd.Index(test_labels.tolist() +  train_labels.tolist())

In [15]:
from scipy.sparse import coo_matrix

In [16]:
aug_all = coo_matrix(aug_mat.loc[labels,labels].values )

In [17]:
ct_codes = np.array([0]*len(test_labels) + ct_codes)

In [18]:
def preprocess(train, test):
    labels = np.zeros((test.shape[0], ), dtype=int)
    data = test
    for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data = np.append(data, examples, axis=0)
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    return data, labels

In [19]:
data, labels = preprocess(train_data, test_data)

In [20]:
from sklearn.neighbors import NearestNeighbors
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel

In [21]:
def create_graph(data, k=30, metric='euclidean', n_jobs=-1):
    # def _kernel(dxy, sigma=1):
    #     return np.exp(-dxy ** 2 / sigma)

    _, idx = find_neighbors(data, k=k, metric=metric, n_jobs=n_jobs)
    # affinities = np.apply_along_axis(lambda x: _kernel(x, x.std()), axis=1, arr=d)
    # n, k = idx.shape
    # i = [np.tile(x, (k, )) for x in range(n)]
    # i = np.concatenate(np.array(i))
    # j = np.concatenate(idx)
    # s = np.concatenate(affinities)
    # graph = sp.coo_matrix((s, (i, j)), shape=(n, n)).tocsr()
    # graph = (graph + graph.transpose()).multiply(.5)
    graph = neighbor_graph(jaccard_kernel, {'idx': idx})
    # make symmetric
    # graph = (graph + graph.transpose()).multiply(.5)
    return graph

In [22]:
# Assuming aug_all is your coo_matrix
subset_rows = slice(0, 50)
subset_cols = slice(0, 10)
subset_aug_all = aug_all.tocsr()[subset_rows, subset_cols]

# Now pass the subset to the create_graph function
create_graph(subset_aug_all)

dense_subset_aug_all = subset_aug_all.toarray()
create_graph(dense_subset_aug_all)

Finding 30 nearest neighbors using minkowski metric and 'auto' algorithm


TypeError: sparse array length is ambiguous; use getnnz() or shape[0]

In [23]:
dense_subset_aug_all = subset_aug_all.toarray()
create_graph(dense_subset_aug_all)

Finding 30 nearest neighbors using minkowski metric and 'auto' algorithm


<50x50 sparse matrix of type '<class 'numpy.float64'>'
	with 1500 stored elements in COOrdinate format>

In [24]:
import numpy as np
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
import scipy.sparse as sp
from sklearn.preprocessing import normalize


def random_walk_probabilities(A, labels):

    if labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()
        D = sp.diags(A.sum(axis=1).flatten(), [0], shape=A.shape, format='csr')
        L = D - A
        seeds = labels != 0
        Lu = L[np.invert(seeds), :]  # unlabeled rows
        Lu = Lu.tocsc()[:, np.invert(seeds)]  # unlabeled columns
        # Check that Lu has the right size
        if not all([t == sum(np.invert(seeds)) for t in Lu.shape]):
            raise IndexError("Lu should be square and match size of test data")
        BT = L[np.invert(seeds), :]  # unlabeled rows
        BT = BT.tocsc()[:, seeds]  # labeled columns
        if not (sum(np.invert(seeds)), sum(seeds)) == BT.shape:
            raise IndexError("BT size is incorrect")
        # Matrix representation of labels
        i, j, s = [], [], []
        classes = np.unique(labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(labels[seeds] == k)[0])
            j.extend(np.tile(k, sum(seeds == k)))
            s.extend(np.tile(1, sum(seeds == k)))
        i = np.arange(seeds.sum())
        j = labels[seeds] - 1
        s = np.ones((seeds.sum(), ))
        M = sp.coo_matrix((s, (i, j)), shape=(seeds.sum(), len(classes))).tocsc()
        # P = sp.linalg.spsolve(Lu.tocsc(), -BT.dot(M))
        # Use iterative solver
        B = -BT.dot(M)
        vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]
        warnings = [x[1] for x in vals]
        if any(warnings):
            print("Warning: iterative solver failed to converge in at least one case", flush=True)
        P = normalize(np.vstack(tuple((x[0] for x in vals))).T, norm='l1')

    else:
        D = np.diag(np.sum(A, axis=1))
        L = D - A  # graph laplacian
        seeds = np.array([e != 0 for e in labels], dtype=bool)
        Lu = L[seeds, :][:, seeds]  # labeled rows, labeled cols
        BT = L[~seeds, :][:, seeds]  # unlabeled rows, labeled cols
        classes = np.unique(labels[labels > 0])
        M = np.zeros((seeds.sum(), len(classes)))
        for k in classes:
            M[labels[seeds] == k, k] = 1
        P = np.linalg.lstsq(Lu, np.dot(-BT, M))[0]

    return P

In [25]:
P = random_walk_probabilities(aug_all, ct_codes)

/scratch/lsftmp/3030658.tmpdir/ipykernel_68933/3655095487.py:42: DeprecationWarning: Please use `bicgstab` from the `scipy.sparse.linalg` namespace, the `scipy.sparse.linalg.isolve` namespace is deprecated.
  vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]


In [26]:
pval_df = pd.DataFrame(P,
                           index = test_cells.obs_names,
                           columns = cell_types)

In [27]:
pval_df
#pval_df.to_csv('/data/chanjlab/forsythb/phenograph_classification_8.csv', index=True)
pval_df.to_csv('/data/chanjlab/forsythb/phenograph_classification_9.csv', index=True) # This is the one with BASE and HISC

In [28]:
pval_df

,ISC-like,SCC,Fetal Progenitor,Enterocyte-like,TA-like,Injury Repair,Early NET,Goblet-like
146P_BASE_shZFP36L2_3_TTAGGGTCAGTAACGG-1,0.247062,0.023131,0.013075,0.119771,0.440876,0.082291,0.002830,0.070963
KG146Li_BASE_shZFP36L2_4_CGTTAGACACGTAACT-1,0.201914,0.071850,0.074118,0.140977,0.202472,0.270251,0.006091,0.032326
146P_BASE_shCTRL_AGGTGTTCATACAGGG-1,0.257828,0.028085,0.015086,0.114352,0.438303,0.094121,0.002595,0.049631
146P_BASE_shZFP36L2_4_CCTCTAGGTCGCATTA-1,0.255894,0.029284,0.014910,0.105886,0.447552,0.097014,0.002494,0.046967
125P_HISC_shZFP36L2_3_GTTGTCCTCCACTGGG-1,0.384065,0.010555,0.006582,0.377946,0.032471,0.073220,0.000778,0.114383
...,...,...,...,...,...,...,...,...
125P_HISC_shZFP36L2_4_TTCGGTCTCCGTTGGG-1,0.383257,0.011600,0.007240,0.372824,0.035258,0.079105,0.000845,0.109871
146P_HISC_shZFP36L2_4_ACAAAGATCATCACTT-1,0.322675,0.029498,0.020252,0.289435,0.191517,0.107792,0.002172,0.036659
146P_BASE_shZFP36L2_3_TCAGGTAAGTTTAGGA-1,0.251893,0.037920,0.020122,0.127675,0.396350,0.115804,0.003045,0.047191
146P_HISC_shZFP36L2_3_TCATCATAGATCGGTG-1,0.330516,0.026202,0.017472,0.307102,0.186499,0.093520,0.001947,0.036743


### Plotting Ternary Plots

# Phenograph Classification

In [3]:
import phenograph

In [4]:
adata_patient_organoid = sc.read_h5ad('/data/chanjlab/forsythb/adata_patient_organoid.h5ad')

In [45]:
'''
Here, there are 9092 NA values. The 'Cell State' observations
are the original annotations for patient tumors. The NA values 
correpond to the organoid data. 
'''
adata_patient_organoid.obs['Cell State'].value_counts()

Cell State
NA                  9092
nan                  928
SCC                  280
ISC-like             252
Injury Repair        230
Enterocyte-like      223
Fetal Progenitor     106
TA-like               80
Goblet-like           71
Early NET             62
Name: count, dtype: int64

In [43]:
'''
In the 'PhenoGraph Classification' observation, there are 2232 nan values. 
This observation is the original annotation for organoid data. The nan values
correspond to the patient tumor data. 
'''
adata_patient_organoid.obs['PhenoGraph Classification'].value_counts()

PhenoGraph Classification
Enterocyte-like     3201
nan                 2232
TA-like             1913
ISC-like            1770
Injury Repair       1160
Goblet-like          371
Fetal Progenitor     318
SCC                  295
Early NET             64
Name: count, dtype: int64

In [48]:
''' 
The nan values correspond to the patient tumor data, so only
the patient metastatic data is used in the analysis. We will
drop those values. Also, we are only interested in organoid  
classification, which corresponds to 'Cell State' == NA.
'''
adata_patient_organoid.obs['Cell State'].replace('nan', np.nan, inplace=True)
adata_patient_organoid.obs.dropna(subset=['Cell State'], inplace=True)

In [49]:
'''
Separate training and testing sets based on 'Cell State' column.
The test data is the organoid data that we want to classify, which 
corresponds to 'Cell State' == NA. 
'''
train_adata = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'NA']
test_adata = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] == 'NA']

In [100]:
def preprocess(train, test):
    labels = np.zeros((test.shape[0], ), dtype=int)
    data = test
    for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data = np.append(data, examples, axis=0)
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    return data, labels

In [ ]:
n_neighbors = 60
ct_metacluster_preds_dm = phenograph.classify(train_adata, test_adata, k=n_neighbors, metric='euclidean')

In [97]:
cell_types = sorted(train_adata.obs['Cell State'].cat.categories)

dm_merge = pd.DataFrame(test_adata.obsm['aug_mat'], index=test_adata.obs.index).loc[:, :]
#dm_merge = test_adata
train = np.empty((len(cell_types),),dtype=object)

for c, cell_type in enumerate(cell_types):
    labels = train_adata.obs.index[train_adata.obs['Cell State'] == cell_type]
    #print(f'Cell Type: {cell_type}, Labels: {labels}')
    ind = [train_adata.obs.index.get_loc(i) for i in labels]
    train[c] = dm_merge.iloc[ind, :]
    #train[c] = dm_merge[ind, :]
    
test = dm_merge

In [78]:
def classify(train, test, k=30, metric='euclidean', n_jobs=-1):
    """
    Semi-supervised classification by random walks on a graph
    :param train: list of numpy arrays. Each array has a row for each class observation
    :param test: numpy array of unclassified data
    :return c: class assignment for each row in test
    :return P: class probabilities for each row in test
    """
    #data, labels = preprocess(train, test)
    # Build graph
    #A = create_graph(data, k, metric=metric, n_jobs=n_jobs)
    A = train_adata.obsm['aug_mat']
    P = random_walk_probabilities(A, labels)
    c = np.argmax(P, axis=1)
    return c, P

In [99]:
n_neighbors = 60
ct_metacluster_preds_dm = phenograph.classify(train_adata, test_adata, k=n_neighbors, metric='euclidean')

Preprocessing:   0%|          | 0/1304 [58:42<?, ? example(s)/s]
Exception ignored in: <function _xla_gc_callback at 0x2b6070452790>
Traceback (most recent call last):
  File "/home/forsythb/.local/lib/python3.9/site-packages/jax/_src/lib/__init__.py", line 101, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


KeyboardInterrupt: 

In [94]:
def random_walk_probabilities(A, labels):

    # Convert labels to start from 0
    unique_labels = np.unique(labels)
    label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
    encoded_labels = np.array([label_mapping[label] for label in labels])

    if encoded_labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()
        D = sp.diags(A.sum(axis=1).flatten(), [0], shape=A.shape, format='csr')
        L = D - A
        seeds = encoded_labels != 0
        Lu = L[np.invert(seeds), :]  # unlabeled rows
        Lu = Lu[:, np.invert(seeds)]  # unlabeled columns
        if not all([t == sum(np.invert(seeds)) for t in Lu.shape]):
            raise IndexError("Lu should be square and match size of test data")
        BT = L[np.invert(seeds), :]  # unlabeled rows
        BT = BT[:, seeds]  # labeled columns
        if not (sum(np.invert(seeds)), sum(seeds)) == BT.shape:
            raise IndexError("BT size is incorrect")
        i, j, s = [], [], []
        classes = np.unique(encoded_labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(encoded_labels[seeds] == k)[0])
            j.extend(np.tile(k, sum(seeds == k)))
            s.extend(np.tile(1, sum(seeds == k)))
        i = np.arange(seeds.sum())
        j = encoded_labels[seeds] - 1
        s = np.ones((seeds.sum(), ))
        M = sp.coo_matrix((s, (i, j)), shape=(seeds.sum(), len(classes))).tocsc()
        vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in -BT.dot(M).T]
        warnings = [x[1] for x in vals]
        if any(warnings):
            print("Warning: iterative solver failed to converge in at least one case", flush=True)
        P = normalize(np.vstack(tuple((x[0] for x in vals))).T, norm='l1')

    else:
        D = np.diag(A.sum(axis=1))  # Ensure D is diagonal
        L = D - A  # graph laplacian
        seeds = np.array([e != 0 for e in encoded_labels], dtype=bool)
        Lu = L[np.invert(seeds), np.invert(seeds)]  # unlabeled rows, unlabeled cols
        BT = L[np.invert(seeds), seeds]  # unlabeled rows, labeled cols
        classes = np.unique(encoded_labels[encoded_labels > 0])
        M = np.zeros((seeds.sum(), len(classes)))
        for k in classes:
            M[encoded_labels[seeds] == k, k] = 1
        P = np.linalg.lstsq(Lu, np.dot(-BT, M))[0]

    return P



In [95]:
n_neighbors = 60
ct_metacluster_preds_dm = classify(train_adata, test_adata, k=n_neighbors, metric='euclidean')

ValueError: operands could not be broadcast together with shapes (1304,1304) (1304,11324) 

In [73]:
pval_df = pd.DataFrame(ct_metacluster_preds_dm[1],
                           index = test.index,
                           columns = cell_types)

pval_df.to_csv('/data/chanjlab/forsythb/phenograph_classification_3.csv', index=True)

In [62]:
def classify(train_adata, test_adata, k=30, metric='euclidean', n_jobs=-1):
    """
    Semi-supervised classification by random walks on a graph
    :param train_adata: AnnData object with labeled training data
    :param test_adata: AnnData object with unlabeled testing data
    :return c: class assignment for each row in test
    :return P: class probabilities for each row in test
    """
    # Extract labels and preprocess the data
    data, labels = preprocess(train_adata, test_adata)

    # Use augmented affinity matrix from adata_patient_organoid.obsm['aug_mat']
    A = adata_patient_organoid.obsm['aug_mat']

    # Ensure that the affinity matrix has the correct shape
    if A.shape[0] != A.shape[1] or A.shape[0] != len(labels):
        raise ValueError("Mismatch in the shape of the affinity matrix and labels.")

    # Compute random walk probabilities
    P = random_walk_probabilities(A, labels)

    # Class assignment based on the highest probability
    c = np.argmax(P, axis=1)

    return c, P

In [59]:
import numpy as np
from tqdm import tqdm

def preprocess(train, test):
    num_train = len(train)
    num_test = test.shape[0]

    # Initialize lists to accumulate labels and data
    labels_list = []
    data_list = []

    # Create a tqdm progress bar
    progress_bar = tqdm(total=num_train, desc="Preprocessing", unit=" example(s)")

    # Iterate over training examples
    for c, examples in enumerate(train):
        labels_list.append(np.tile(c + 1, (examples.shape[0],)))
        data_list.append(examples)

        # Update the progress bar
        progress_bar.update()

    # Close the progress bar
    progress_bar.close()

    # Convert lists to numpy arrays
    labels = np.concatenate(labels_list)
    data = np.concatenate(data_list, axis=0)

    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != num_test:
        raise IndexError("Labels should include one 0 for every row of test data")

    return data, labels

In [60]:
data, labels = preprocess(train_adata, test_adata)



Preprocessing:   0%|          | 0/1304 [00:00<?, ? example(s)/s]

Preprocessing:   0%|          | 3/1304 [00:00<00:48, 27.02 example(s)/s]

Preprocessing:   1%|          | 7/1304 [00:00<00:40, 32.01 example(s)/s]

Preprocessing:   1%|          | 11/1304 [00:00<00:39, 32.80 example(s)/s]

Preprocessing:   1%|          | 15/1304 [00:00<00:38, 33.46 example(s)/s]

Preprocessing:   1%|▏         | 19/1304 [00:00<00:43, 29.48 example(s)/s]

Preprocessing:   2%|▏         | 23/1304 [00:00<00:41, 30.75 example(s)/s]

Preprocessing:   2%|▏         | 27/1304 [00:00<00:40, 31.86 example(s)/s]

Preprocessing:   2%|▏         | 31/1304 [00:00<00:39, 32.36 example(s)/s]

Preprocessing:   3%|▎         | 35/1304 [00:01<00:39, 32.44 example(s)/s]

Preprocessing:   3%|▎         | 39/1304 [00:01<00:37, 33.41 example(s)/s]

Preprocessing:   3%|▎         | 43/1304 [00:01<00:36, 34.63 example(s)/s]

Preprocessing:   4%|▎         | 48/1304 [00:01<00:34, 36.49 example(s)/s]

Preprocessing:   4%|▍         | 52

Preprocessing:  63%|██████▎   | 816/1304 [00:22<00:13, 36.97 example(s)/s]

Preprocessing:  63%|██████▎   | 820/1304 [00:22<00:12, 37.46 example(s)/s]

Preprocessing:  63%|██████▎   | 824/1304 [00:22<00:12, 37.74 example(s)/s]

Preprocessing:  63%|██████▎   | 828/1304 [00:22<00:13, 35.38 example(s)/s]

Preprocessing:  64%|██████▍   | 832/1304 [00:22<00:12, 36.64 example(s)/s]

Preprocessing:  64%|██████▍   | 836/1304 [00:23<00:12, 37.43 example(s)/s]

Preprocessing:  64%|██████▍   | 840/1304 [00:23<00:12, 38.13 example(s)/s]

Preprocessing:  65%|██████▍   | 844/1304 [00:23<00:12, 38.20 example(s)/s]

Preprocessing:  65%|██████▌   | 848/1304 [00:23<00:11, 38.63 example(s)/s]

Preprocessing:  65%|██████▌   | 852/1304 [00:23<00:11, 38.91 example(s)/s]

Preprocessing:  66%|██████▌   | 856/1304 [00:23<00:11, 38.50 example(s)/s]

Preprocessing:  66%|██████▌   | 860/1304 [00:23<00:11, 38.90 example(s)/s]

Preprocessing:  66%|██████▋   | 864/1304 [00:23<00:11, 39.21 example(s)/s]

Preprocessin

ValueError: setting an array element with a sequence. The requested array would exceed the maximum number of dimension of 32.

In [33]:
# Get true labels for all categories except 'nan'
true_labels.replace('nan', np.nan, inplace=True)
true_labels.value_counts()

Cell State
NA                  9092
SCC                  280
ISC-like             252
Injury Repair        230
Enterocyte-like      223
Fetal Progenitor     106
TA-like               80
Goblet-like           71
Early NET             62
Name: count, dtype: int64

In [5]:
# Assuming 'Cell State' is the column containing the true cell types in your dataset
true_labels = adata_patient_organoid.obs['Cell State']

# Filter out rows with nan labels
true_labels_no_nan = true_labels.dropna()

# Remove 'nan' from unique values
cell_types = np.sort(true_labels_no_nan.unique())
cell_types = np.setdiff1d(cell_types, ['nan'])

dm_merge = pd.DataFrame(adata_patient_organoid.obsm['aug_mat'], index=adata_patient_organoid.obs.index).loc[:, :]

train = np.empty((len(cell_types),), dtype=object)

for c, cell_type in enumerate(cell_types):
    labels = true_labels_no_nan[true_labels_no_nan == cell_type].index
    ind = [adata_patient_organoid.obs.index.get_loc(i) for i in labels]
    train[c] = dm_merge.loc[labels, :].copy()

test = dm_merge

In [6]:
# Assuming 'Cell State' is the column containing the true cell types in your dataset
true_labels = adata_patient_organoid.obs['Cell State']

# Iterate through each cell type in the training set and compare the obtained labels
for c, cell_type in enumerate(cell_types):
    # Extract labels for the current cell type
    obtained_labels = adata_patient_organoid.obs.index[adata_patient_organoid.obs['Cell State'] == cell_type]

    # Check if obtained labels match the true labels
    if not np.array_equal(obtained_labels, true_labels[true_labels == cell_type].index):
        print(f"Mismatch for cell type {cell_type}. Check labels for correctness.")
    else:
        print(f"Labels for cell type {cell_type} are correct.")

Labels for cell type Early NET are correct.
Labels for cell type Enterocyte-like are correct.
Labels for cell type Fetal Progenitor are correct.
Labels for cell type Goblet-like are correct.
Labels for cell type ISC-like are correct.
Labels for cell type Injury Repair are correct.
Labels for cell type NA are correct.
Labels for cell type SCC are correct.
Labels for cell type TA-like are correct.


In [7]:
n_neighbors = 60
ct_metacluster_preds_dm = phenograph.classify(train, test, k=n_neighbors, metric='euclidean')

Finding 60 nearest neighbors using minkowski metric and 'auto' algorithm


In [8]:
pval_df = pd.DataFrame(ct_metacluster_preds_dm[1],
                           index = test.index,
                           columns = cell_types)

In [9]:
pval_df.to_csv('/data/chanjlab/forsythb/phenograph_classification_new.csv', index=True)

In [8]:
pval_df

,Early NET,Enterocyte-like,Fetal Progenitor,Goblet-like,ISC-like,Injury Repair,NA,SCC,TA-like,nan
120703408859571_OKG146P_Base,2.990476e-07,0.034247,6.767366e-07,0.000073,0.001938,0.000043,0.857647,2.999503e-07,0.000022,0.106029
120703423732149_OKG146P_Base,1.213028e-06,0.031450,6.380103e-07,0.000166,0.002179,0.000041,0.878625,8.031547e-07,0.000019,0.087516
120703423986909_OKG146P_Base,2.953009e-07,0.033807,6.846186e-07,0.000074,0.001959,0.000043,0.854226,3.034517e-07,0.000022,0.109868
120703436417774_OKG146P_Base,2.615074e-07,0.030163,5.913392e-07,0.000064,0.001694,0.000037,0.907919,2.616626e-07,0.000019,0.060103
120718456404405_OKG146P_Base,3.157573e-07,0.036148,7.179533e-07,0.000078,0.002055,0.000046,0.848599,3.183740e-07,0.000024,0.113049
...,...,...,...,...,...,...,...,...,...,...
165190304708846_KG146P,3.014074e-07,0.034068,1.182991e-06,0.000078,0.023785,0.000079,0.696443,6.912458e-07,0.000023,0.245522
161384145836979_KG146P,2.904508e-07,0.033212,8.626878e-07,0.000105,0.002787,0.000062,0.756708,3.342778e-07,0.000023,0.207102
200897607945508_KG146P,2.791208e-07,0.031591,6.325120e-07,0.000069,0.002421,0.000050,0.535648,2.851363e-07,0.000020,0.430199
125033721486630_KG146P,2.676165e-07,0.031091,6.206819e-07,0.000068,0.002900,0.000040,0.470007,2.731663e-07,0.000024,0.495868


In [10]:
pval_df.to_csv('/data/chanjlab/forsythb/phenograph_classification.csv', index=True)

In [76]:
# Count occurrences of each unique value
unique_values, counts = np.unique(ct_metacluster_preds_dm[0], return_counts=True)

# Display the counts
for value, count in zip(unique_values, counts):
    print(f"{value}: {count} occurrences")

0: 18 occurrences
1: 31 occurrences
2: 9 occurrences
3: 19 occurrences
4: 11 occurrences
5: 31 occurrences
6: 10950 occurrences
7: 32 occurrences
8: 8 occurrences
9: 215 occurrences


In [24]:
for c,cell_type in enumerate(cell_types):
    labels = WT_cell_type.index[WT_cell_type == cell_type]
    ind = [adata.obs.index.get_loc(i) for i in labels]
    train[c] = dm_merge.iloc[ind,:]

array([None, None, None, None, None, None, None, None, None, None],
      dtype=object)

In [ ]:
cell_types = sorted(WT_cell_type.cat.categories)

dm_merge = pd.DataFrame(adata.obsm[diffmap_rep],
                        index = adata.obs.index).loc[:,:num_dcs]

train = np.empty((len(cell_types),),dtype=object)
for c,cell_type in enumerate(cell_types):
    labels = WT_cell_type.index[WT_cell_type == cell_type]
    ind = [adata.obs.index.get_loc(i) for i in labels]
    train[c] = dm_merge.iloc[ind,:]
test = dm_merge 

ct_metacluster_preds_dm = phenograph.classify(train, test, k=n_neighbors, metric='euclidean')

pval_df = pd.DataFrame(ct_metacluster_preds_dm[1],
                       index = test.index,
                       columns = cell_types)

In [15]:
cell_types = adata_patient_organoid.obs['Cell State'].unique()

In [16]:
cell_types

['NA', 'ISC-like', 'SCC', 'Fetal Progenitor', 'Enterocyte-like', 'TA-like', 'Injury Repair', 'Early NET', 'Goblet-like', 'nan']
Categories (10, object): ['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', ..., 'NA', 'SCC', 'TA-like', 'nan']